# Qwen LoRA fine-tuning and evaluation

LRZ/Colab GPU session required.

## Run order

1. Run the dependency cell, then restart the kernel if packages changed
2. Edit the **config cell** below, then run it
3. Run the model cell, then the training cell, then the evaluation cell

## Modifiable settings (config cell)

| Variable | Description |
|----------|-------------|
| `HF_TOKEN` | Hugging Face token (optional for public models); read from the `HF_TOKEN` env var |
| `MODE` | `None` = all modes (train + eval); or `"no_term"`, `"proper_term"`, `"random_term"` |
| `TRAIN_DATA_DIR` | Folder with training JSONL (e.g. `experiments/04_lora_finetuning/data/training`) |
| `TEST_DATA_DIR` | Folder with test JSONL (e.g. `experiments/04_lora_finetuning/data/test`) |
| `OUTPUT_DIR` | Predictions, metrics, and LoRA adapter (`OUTPUT_DIR/adapter/`) |
| `MAX_TRAIN_SAMPLES` | `None` = all training samples |
| `MAX_TEST_SAMPLES` | `None` = all test samples |
| `USE_FEW_SHOT` | Include 3 hardcoded few-shot examples in train and eval prompts |
| `NUM_EPOCHS` | LoRA training epochs |
| `MODEL_NAME` | Hugging Face model id |

`MODE=None` triples training examples (one pass per mode). Paths are relative to the kernel working directory.

## Inputs / outputs

Inputs: one JSONL per language in each data dir, e.g. `ende_dev_v1_training.jsonl`, `ende_dev_v1_test.jsonl`.

Outputs: `OUTPUT_DIR/adapter/`, `OUTPUT_DIR/{lang}/*_{mode}_predictions.jsonl`, `OUTPUT_DIR/metrics_summary.json`.

In [ ]:
# Pin transformers: LRZ PyTorch (nv24.08) lacks torch.float8_e8m0fnu required by transformers>=4.51.
%pip install -q "transformers>=4.46,<4.51" accelerate peft sacrebleu tqdm huggingface_hub ipywidgets
print("Restart kernel, then run from the config cell.")

In [ ]:
from pathlib import Path
import os

WORK_DIR = Path.cwd()

# --- user settings ---

# Set via `HF_TOKEN` env var or a .env file; optional for public models.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
MODE = "proper_term"  # None | "no_term" | "proper_term" | "random_term"
# TRAIN_DATA_DIR = "/dss/dsshome1/0E/go35bit2/experiments/04_lora_finetuning/data/training"
# TEST_DATA_DIR = "/dss/dsshome1/0E/go35bit2/experiments/04_lora_finetuning/data/test"
# OUTPUT_DIR = "/dss/dsshome1/0E/go35bit2/experiments/04_lora_finetuning/results/qwen_lora"
TRAIN_DATA_DIR = "data/training"
TEST_DATA_DIR = "data/test"
OUTPUT_DIR = "results/qwen_lora"
MAX_TRAIN_SAMPLES = None
MAX_TEST_SAMPLES = None
USE_FEW_SHOT = True
NUM_EPOCHS = 1
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# --- resolved paths and modes ---

ALL_MODES = ["no_term", "proper_term", "random_term"]
LANG_GROUPS = ["ende", "enru", "enes"]

if MODE is None:
    MODES = ALL_MODES
else:
    if MODE not in ALL_MODES:
        raise ValueError(f"MODE must be None or one of {ALL_MODES}")
    MODES = [MODE]

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()

TRAIN_DATA_ROOT = Path(TRAIN_DATA_DIR).expanduser().resolve()
TEST_DATA_ROOT = Path(TEST_DATA_DIR).expanduser().resolve()
if not TRAIN_DATA_ROOT.is_dir():
    raise FileNotFoundError(f"TRAIN_DATA_DIR not found: {TRAIN_DATA_ROOT}")
if not TEST_DATA_ROOT.is_dir():
    raise FileNotFoundError(f"TEST_DATA_DIR not found: {TEST_DATA_ROOT}")

OUTPUT_BASE = Path(OUTPUT_DIR).expanduser().resolve()
ADAPTER_DIR = OUTPUT_BASE / "adapter"

LANG_CONFIG = {
    "ende": {"ref_field": "de", "target_lang": "German", "output_tag": "de"},
    "enru": {"ref_field": "ru", "target_lang": "Russian", "output_tag": "ru"},
    "enes": {"ref_field": "es", "target_lang": "Spanish", "output_tag": "es"},
}

SAMPLE_SENTENCES: dict[str, list[dict[str, object]]] = {
    "ende": [
        {"en": "This service describes the deployed (run-time) state of SAP HANA database artifacts, for example: tables, views, or procedures, which have been created or adjusted by the SAP Integrated Development Environment (WebIDE) editors as a family of consistent design-time artifacts for all key SAP HANA platform database features.\n", "de": "Dieser Service beschreibt den implementierten Zustand (Laufzeitzustand) von SAP-HANA-Datenbankartefakten, z. B. Tabellen, Views oder Prozeduren, die von den SAP-Integrated-Development-Environment-Editoren (WebIDE-Editoren) als eine Familie konsistenter Entwurfszeit-Artefakte für alle wichtigen SAP-HANA-Plattform-Datenbankfunktionen erstellt oder angepasst wurden.\n", "proper_terms": {"design": "Entwurf", "state": "Zustand"}, "random_terms": {"artifacts": "Artefakten", "key": "wichtigen"}},
        {"en": "Request a checkup to perform a health diagnostic and better analyze what was wrong with your data flow.\n", "de": "Fordern Sie eine Kontrolle an, um eine Fehlerdiagnose durchzuführen und besser zu analysieren, was mit Ihrem Datenfluss falsch war.\n", "proper_terms": {"check": "Kontrolle"}, "random_terms": {"wrong": "falsch"}},
        {"en": "The data product is still available in the provider's data product list.\n", "de": "Das Datenprodukt ist weiterhin in der Datenproduktliste des Providers verfügbar.\n", "proper_terms": {"provider": "Provider"}, "random_terms": {"'s": "des"}}
    ],
    "enru": [
        {"en": "Indicates if a configuration item or configuration step is specific to a localized solution version.\n", "ru": "Указывает, являются ли позиция или шаг конфигурации специфичными для локализованной версии решения.\n", "proper_terms": {"item": "позиция"}, "random_terms": {"localized": "локализованной"}},
        {"en": "You run allocation cycles in the Run Allocations app.\n", "ru": "Для выполнения циклов перерасчета используется приложение Выполнить перерасчеты.\n", "proper_terms": {"run": "выполнить"}, "random_terms": {"cycles": "циклов"}},
        {"en": "Depending on your use case, you can choose between the following types of allocations:\n", "ru": "В зависимости от варианта использования можно выбрать один из следующих типов перерасчета:\n", "proper_terms": {"type": "тип"}, "random_terms": {"choose": "выбрать"}}
    ],
    "enes": [
        {"en": "In such cases you may use the Move Items or Merge feature.\n", "es": "En estos casos, puede utilizar la función Mover elementos o Fusionar .\n", "proper_terms": {"item": "elemento"}, "random_terms": {"Move": "Mover"}},
        {"en": "Save and Publish\n", "es": "Guardar y publicar\n", "proper_terms": {"save": "guardar"}, "random_terms": {"Publish": "publicar"}},
        {"en": "Required Permissions for SQL Server Trigger-Based Replication in the SAP HANA Smart Data Integration and SAP HANA Smart Data Quality Installation and Configuration Guide\n", "es": "Permisos necesarios para reproducción basada en desencadenador de SQL Server en la guía de instalación y configuración de Integración de datos inteligentes de SAP HANA y calidad de los datos inteligentes de SAP HANA\n", "proper_terms": {"replication": "reproducción"}, "random_terms": {"SQL": "SQL"}}
    ],
}


def discover_data_files(data_root: Path) -> dict[str, Path]:
    files: dict[str, Path] = {}
    for lang in LANG_GROUPS:
        matches = sorted(data_root.glob(f"{lang}_*.jsonl"))
        if not matches:
            raise FileNotFoundError(f"No JSONL for {lang} in {data_root}")
        if len(matches) > 1:
            raise ValueError(f"Multiple JSONL files for {lang}: {matches}")
        files[lang] = matches[0]
    return files


TRAIN_FILES = discover_data_files(TRAIN_DATA_ROOT)
TEST_FILES = discover_data_files(TEST_DATA_ROOT)


def train_data_path(lang: str) -> Path:
    return TRAIN_FILES[lang]


def test_data_path(lang: str) -> Path:
    return TEST_FILES[lang]


def prediction_stem(lang: str) -> str:
    return TEST_FILES[lang].stem


print("KERNEL_CWD:", WORK_DIR)
print("TRAIN_DATA_DIR:", TRAIN_DATA_ROOT)
print("TEST_DATA_DIR:", TEST_DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_BASE)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("MODE:", MODE, "->", MODES)
print("USE_FEW_SHOT:", USE_FEW_SHOT)
print("NUM_EPOCHS:", NUM_EPOCHS)
print("MODEL_NAME:", MODEL_NAME)
print("MAX_TRAIN_SAMPLES:", "all" if MAX_TRAIN_SAMPLES is None else MAX_TRAIN_SAMPLES)
print("MAX_TEST_SAMPLES:", "all" if MAX_TEST_SAMPLES is None else MAX_TEST_SAMPLES)
for label, files in [("train", TRAIN_FILES), ("test", TEST_FILES)]:
    for lang in LANG_GROUPS:
        print(f"  {label}/{lang}: {files[lang].name} [ok]")

In [ ]:
import os
from pathlib import Path

import torch

for _fp8_name in ("float8_e8m0fnu", "float8_e4m3fn", "float8_e5m2"):
    if not hasattr(torch, _fp8_name):
        setattr(torch, _fp8_name, torch.float32)

from transformers import AutoModelForCausalLM, AutoTokenizer


def setup_hf_cache(root: Path) -> Path:
    if os.environ.get("HF_HOME"):
        cache_root = Path(os.environ["HF_HOME"]).expanduser()
    elif os.environ.get("SCRATCH"):
        cache_root = Path(os.environ["SCRATCH"]) / "huggingface_cache"
    else:
        cache_root = root / ".cache" / "huggingface"
    cache_root.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(cache_root)
    os.environ["HUGGINGFACE_HUB_CACHE"] = str(cache_root / "hub")
    return cache_root


HF_CACHE = setup_hf_cache(Path.cwd())

print("HF cache:", HF_CACHE)
print("torch:", torch.__version__)
if torch.cuda.is_available():
    print("CUDA:", torch.cuda.get_device_name(0))
else:
    print("Warning: no GPU — CPU only (very slow).")

load_kwargs = {
    "torch_dtype": "auto",
    "cache_dir": str(HF_CACHE),
    "device_map": "auto" if torch.cuda.is_available() else "cpu",
}

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=str(HF_CACHE))
print("Model ready.")

In [ ]:
import json
from typing import Any

from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_LEARNING_RATE = 2e-4
LORA_PER_DEVICE_TRAIN_BATCH_SIZE = 1
LORA_GRADIENT_ACCUMULATION_STEPS = 8
LORA_MAX_SEQ_LENGTH = 1024


def load_jsonl(path: Path, max_samples: int | None = None) -> list[dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
            if max_samples is not None and len(records) >= max_samples:
                break
    return records


def load_train_datasets() -> dict[str, list[dict[str, Any]]]:
    return {lang: load_jsonl(train_data_path(lang), MAX_TRAIN_SAMPLES) for lang in LANG_GROUPS}


def terms_for_mode(sample: dict[str, Any], mode: str) -> dict[str, str]:
    if mode == "proper_term":
        return (sample.get("proper_terms") or {}).copy()
    if mode == "random_term":
        terms = (sample.get("random_terms") or {}).copy()
        for key in (sample.get("proper_terms") or {}):
            terms.pop(key, None)
        return terms
    return {}


def terminology_for_mode(sample: dict[str, Any], mode: str) -> dict[str, str] | None:
    terms = terms_for_mode(sample, mode)
    return terms or None


def format_terminology_block(terms: dict[str, str]) -> str:
    if not terms:
        return ""
    return "Terminology:\n" + "\n".join(f"{s} -> {t}" for s, t in terms.items()) + "\n"


def format_sample_examples(lang: str, mode: str) -> str:
    if not USE_FEW_SHOT:
        return ""
    config = LANG_CONFIG[lang]
    ref_field = config["ref_field"]
    output_tag = config["output_tag"]
    blocks = []
    for i, example in enumerate(SAMPLE_SENTENCES[lang], 1):
        term_block = format_terminology_block(terms_for_mode(example, mode))
        ref = example.get(ref_field, "")
        blocks.append(
            f"Example {i}:\n"
            f"{term_block}"
            f"Input:\n<en> {example['en']} </en>\n"
            f"Output:\n<{output_tag}> {ref} </{output_tag}>"
        )
    return "Examples:\n\n" + "\n\n".join(blocks) + "\n\n"


def build_translation_prompt(
    sample_en: str,
    terminology: dict[str, str] | None,
    target_lang: str,
    output_tag: str,
    lang: str,
    mode: str,
) -> str:
    examples_block = format_sample_examples(lang, mode)
    term_block = format_terminology_block(terminology or {})
    if term_block:
        term_block += "\n"
    return f"""You are a translation assistant.

Translate the English text to {target_lang}.

Rules:
1. Output only in this format: <{output_tag}> ... </{output_tag}>
2. Use the terminology mappings exactly as provided.
3. Do not explain anything.
4. Translate only from English to {target_lang}.

{examples_block}{term_block}Input:
<en> {sample_en} </en>
"""


class TranslationPromptDataset(Dataset):
    def __init__(self, datasets_by_lang: dict[str, list[dict[str, Any]]], modes: list[str]):
        self.items: list[tuple[str, dict[str, Any], str]] = []
        for lang, samples in datasets_by_lang.items():
            for mode in modes:
                for sample in samples:
                    self.items.append((lang, sample, mode))

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> dict[str, list[int]]:
        lang, sample, mode = self.items[idx]
        config = LANG_CONFIG[lang]
        ref_field = config["ref_field"]
        output_tag = config["output_tag"]
        prompt = build_translation_prompt(
            sample.get("en", ""),
            terminology_for_mode(sample, mode),
            config["target_lang"],
            output_tag,
            lang,
            mode,
        )
        target = f"<{output_tag}> {sample.get(ref_field, '')} </{output_tag}>"
        prompt_messages = [
            {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ]
        full_messages = prompt_messages + [{"role": "assistant", "content": target}]
        prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
        full_text = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
        encoded = tokenizer(full_text, truncation=True, max_length=LORA_MAX_SEQ_LENGTH)
        prompt_ids = tokenizer(prompt_text, truncation=True, max_length=LORA_MAX_SEQ_LENGTH)["input_ids"]
        labels = encoded["input_ids"].copy()
        prompt_len = min(len(prompt_ids), len(labels))
        labels[:prompt_len] = [-100] * prompt_len
        encoded["labels"] = labels
        return encoded


def causal_lm_collator(features: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    max_len = max(len(feature["input_ids"]) for feature in features)
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for feature in features:
        pad_len = max_len - len(feature["input_ids"])
        batch["input_ids"].append(feature["input_ids"] + [pad_id] * pad_len)
        batch["attention_mask"].append(feature["attention_mask"] + [0] * pad_len)
        batch["labels"].append(feature["labels"] + [-100] * pad_len)
    return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.use_cache = False
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_datasets = load_train_datasets()
train_dataset = TranslationPromptDataset(train_datasets, MODES)
print(f"Training examples: {len(train_dataset)} ({len(MODES)} mode(s) x {sum(len(v) for v in train_datasets.values())} samples)")

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
bf16_enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_enabled = torch.cuda.is_available() and not bf16_enabled
training_args = TrainingArguments(
    output_dir=str(ADAPTER_DIR / "trainer_checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=LORA_PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=LORA_GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LORA_LEARNING_RATE,
    logging_steps=10,
    save_strategy="no",
    bf16=bf16_enabled,
    fp16=fp16_enabled,
    gradient_checkpointing=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=causal_lm_collator,
)
trainer.train()
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
model.config.use_cache = True
print("LoRA adapter saved to:", ADAPTER_DIR)

In [ ]:
import json
import re
from collections import Counter, defaultdict
from typing import Any

import sacrebleu
from peft import PeftModel
from tqdm import tqdm


def load_test_datasets() -> dict[str, list[dict[str, Any]]]:
    return {lang: load_jsonl(test_data_path(lang), MAX_TEST_SAMPLES) for lang in LANG_GROUPS}


def save_jsonl(path: Path, records: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def strip_output_tags(text: str, output_tag: str) -> str:
    if not isinstance(text, str):
        return text
    return re.sub(rf"</?{re.escape(output_tag)}>", "", text, flags=re.IGNORECASE).strip()


def compute_bleu_chrf(hyps: list[str], refs: list[str]) -> dict[str, float]:
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    chrf = sacrebleu.corpus_chrf(hyps, [refs])
    return {"bleu": bleu.score, "chrf": chrf.score}


def _normalize_text(text: str) -> str:
    return " ".join(str(text).lower().split())


def _count_term_occurrences(text: str, term: str) -> int:
    text_norm = _normalize_text(text)
    term_norm = _normalize_text(term)
    return len(re.findall(r"\b" + re.escape(term_norm) + r"\b", text_norm))


def terminology_accuracy(preds: list[str], samples: list[dict[str, Any]], mode: str) -> dict[str, Any]:
    term_ratios: dict[str, float] = {}
    total_terms = 0
    for pred, sample in zip(preds, samples):
        source_text = sample.get("en", "")
        for src, tgt in terms_for_mode(sample, mode).items():
            total_terms += 1
            src_count = max(_count_term_occurrences(source_text, src), 1)
            tgt_count = _count_term_occurrences(pred, tgt)
            term_ratios[src] = min(tgt_count / src_count, 1.0)
    avg_ratio = sum(term_ratios.values()) / len(term_ratios) * 100 if term_ratios else None
    return {"total_terms": total_terms, "avg_ratio_pct": avg_ratio, "per_term_ratios": term_ratios}


def terminology_consistency(preds: list[str], samples: list[dict[str, Any]], mode: str) -> dict[str, Any]:
    term_to_candidates: dict[str, list[str]] = defaultdict(list)
    for pred, sample in zip(preds, samples):
        for src, tgt in terms_for_mode(sample, mode).items():
            candidate = tgt if str(tgt).lower() in str(pred).lower() else "<MISSING>"
            term_to_candidates[src].append(candidate)
    per_term = {}
    macro_scores = []
    weighted_scores = []
    for src, candidates in term_to_candidates.items():
        pseudo_ref = Counter(candidates).most_common(1)[0][0]
        matches = sum(1 for c in candidates if c == pseudo_ref)
        consistency = matches / len(candidates)
        per_term[src] = {
            "occ": len(candidates),
            "pseudo_ref": pseudo_ref,
            "matches": matches,
            "consistency": consistency,
        }
        macro_scores.append(consistency)
        weighted_scores.extend([consistency] * len(candidates))
    return {
        "per_term": per_term,
        "macro_avg_consistency": sum(macro_scores) / len(macro_scores) if macro_scores else None,
        "weighted_avg_consistency": sum(weighted_scores) / len(weighted_scores) if weighted_scores else None,
    }


def fmt_metric(value: float | None, digits: int = 2) -> str:
    return "N/A" if value is None else f"{value:.{digits}f}"


def translate_sample(
    sample_en: str,
    terminology: dict[str, str] | None,
    target_lang: str,
    output_tag: str,
    lang: str,
    mode: str,
    max_new_tokens: int = 256,
) -> str:
    prompt = build_translation_prompt(sample_en, terminology, target_lang, output_tag, lang, mode)
    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.inference_mode():
        generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)
    generated_ids = [out[len(inp):] for inp, out in zip(model_inputs.input_ids, generated_ids)]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()


def prediction_filename(lang: str, mode: str) -> str:
    return f"{prediction_stem(lang)}_{mode}_predictions.jsonl"


def run_mode(
    lang: str,
    mode: str,
    samples: list[dict[str, Any]],
    output_dir: Path,
    config: dict[str, str],
) -> dict[str, Any]:
    ref_field = config["ref_field"]
    preds = []
    records = []
    for sample in tqdm(samples, desc=f"{lang}/{mode}"):
        pred = translate_sample(
            sample.get("en", ""),
            terminology_for_mode(sample, mode),
            config["target_lang"],
            config["output_tag"],
            lang,
            mode,
        )
        preds.append(pred)
        record = sample.copy()
        record[f"prediction_{mode}"] = pred
        record[f"prediction_{mode}_clean"] = strip_output_tags(pred, config["output_tag"])
        records.append(record)
    clean_preds = [strip_output_tags(p, config["output_tag"]) for p in preds]
    pred_path = output_dir / prediction_filename(lang, mode)
    save_jsonl(pred_path, records)
    metrics: dict[str, Any] = {}
    if samples and ref_field in samples[0]:
        refs = [sample.get(ref_field, "") for sample in samples]
        metrics.update(compute_bleu_chrf(clean_preds, refs))
        term_acc = terminology_accuracy(clean_preds, samples, mode)
        term_cons = terminology_consistency(clean_preds, samples, mode)
        metrics["terminology_accuracy"] = term_acc
        metrics["terminology_consistency"] = term_cons
        print(
            f"[{lang}/{mode}] BLEU={fmt_metric(metrics['bleu'])} "
            f"chrF={fmt_metric(metrics['chrf'])} "
            f"term_acc={fmt_metric(term_acc['avg_ratio_pct'])}% "
            f"macro_cons={fmt_metric(term_cons['macro_avg_consistency'])} "
            f"weighted_cons={fmt_metric(term_cons['weighted_avg_consistency'])}"
        )
    else:
        print(f"[{lang}/{mode}] no reference field '{ref_field}' — metrics skipped")
    return {"predictions_file": str(pred_path), "metrics": metrics}


if not isinstance(model, PeftModel):
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval()

datasets = load_test_datasets()
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

summary = {
    "train_data_dir": str(TRAIN_DATA_ROOT),
    "test_data_dir": str(TEST_DATA_ROOT),
    "output_dir": str(OUTPUT_BASE),
    "adapter_dir": str(ADAPTER_DIR),
    "mode": MODE,
    "modes_run": MODES,
    "use_few_shot": USE_FEW_SHOT,
    "num_epochs": NUM_EPOCHS,
    "prompt_examples_per_lang": {lang: len(SAMPLE_SENTENCES[lang]) for lang in LANG_GROUPS},
    "model": MODEL_NAME,
    "max_train_samples": MAX_TRAIN_SAMPLES,
    "max_test_samples": MAX_TEST_SAMPLES,
    "languages": {},
}

for lang in LANG_GROUPS:
    config = LANG_CONFIG[lang]
    samples = datasets[lang]
    lang_dir = OUTPUT_BASE / lang
    lang_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n=== {lang}: {len(samples)} samples → {config['target_lang']} ===")
    lang_results = {mode: run_mode(lang, mode, samples, lang_dir, config) for mode in MODES}
    summary["languages"][lang] = {
        "data_file": str(test_data_path(lang)),
        "sample_count": len(samples),
        **config,
        "modes": lang_results,
    }

metrics_path = OUTPUT_BASE / "metrics_summary.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nDone.", metrics_path)